In [8]:
import os
from dotenv import load_dotenv
import getpass

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
if not os.getenv("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter API key for NVIDIA: ")


In [11]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [12]:
embeddings = NVIDIAEmbeddings(model="nvidia/llama-nemotron-embed-vl-1b-v2")

c:\Users\Atul.Mangla\Desktop\Learning-Langchain\.venv\lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:207: UserWarning: An API key is required for the hosted NIM. This will become an error in the future.
  warnings.warn(
c:\Users\Atul.Mangla\Desktop\Learning-Langchain\.venv\lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:243: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [28]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

video_id = "vOAjCnRbwBQ"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id=video_id)
    # Join transcript text for RAG
    transcript_text = " ".join([entry["text"] for entry in transcript_list])
    print(transcript_text)
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except NoTranscriptFound:
    print("No transcript found for this video.")
except VideoUnavailable:
    print("Video is unavailable.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=vOAjCnRbwBQ (Caused by SSLError(SSLError(1, '[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:1017)')))


In [33]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_text])
print(len(chunks))

chunks[0]

NameError: name 'transcript_text' is not defined

In [ ]:
vectorStore = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vectorStore.as_retriever(search_type="similarity",search_kwargs={"k": 3})

In [ ]:
# Augmentation

prompt = PromptTemplate.from_template(
    """Use the following retrieved information to answer the question. If you don't know the answer, say you don't know.

    {context}

    Question: {question}
    Answer:"""
)


In [ ]:
query = ""

retrieved_docs = retriever.invoke(query)

In [ ]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

final_prompt = prompt.format(context=context_text, question=query)

llm = ChatGroq(model="llama-2-7b-chat-hf")

result = llm.invoke(final_prompt)

print(result)

![improvements for RAG](improvements_RAG.png)

## Building with Chains

In [30]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [31]:
def format_dics(retrieved_docs):
    return "\n\n".join([f"Document {i+1}:\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)])

In [32]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_dics),
    "question": RunnablePassthrough(),
})

NameError: name 'retriever' is not defined

In [ ]:
x = parallel_chain.invoke("")

final_chain = parallel_chain | prompt | llm | StrOutputParser()